# 🍃 Leaf Disease & Quality AI: Training & Edge AI Export
### โครงการนวัตกรรม AIoT และการเกษตรแม่นยำ (CMU AIoT & Scirbru Workshop)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Tsanaphy2023/aiot-workshop/blob/main/leaf_ai_system/Leaf_AI_Training_Colab.ipynb)
[![GitHub Repository](https://img.shields.io/badge/GitHub-aiot--workshop-blue?logo=github)](https://github.com/Tsanaphy2023/aiot-workshop)

สมุดบันทึกนี้ออกแบบสำหรับการฝึกสอน (Train) โมเดลจำแนกโรคพืชและคุณภาพผลผลิต 6 ชนิด (**ข้าวโพด, กาแฟ, มันฝรั่ง, ส้ม, มะม่วง, กล้วย**) ด้วยโมเดล **MobileNetV2 / ResNet18 / LeafNet** โดยใช้การเร่งความเร็วผ่าน **Free GPU (Tesla T4)** บน Google Colab

## 1. ตรวจสอบการเปิดใช้งาน GPU บน Google Colab
> *หมายเหตุ: ไปที่เมนู `Runtime` -> `Change runtime type` แล้วเลือก `T4 GPU`*

In [ ]:
!nvidia-smi

import torch
print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU Device: {torch.cuda.get_device_name(0)}")

## 2. ดึงโค้ดโปรเจกต์จาก GitHub และเตรียมชุดข้อมูล
ดาวน์โหลดโค้ดระบบ `leaf_ai_system` และติดตั้งไลบรารีที่จำเป็น

In [ ]:
import os

# โคลนโปรเจกต์จาก GitHub หากยังไม่มี
if not os.path.exists("aiot-workshop"):
    !git clone https://github.com/Tsanaphy2023/aiot-workshop.git

%cd aiot-workshop/leaf_ai_system
!pip install -q torchvision scikit-learn seaborn matplotlib pandas onnx onnxscript tqdm tabulate

## 3. แตกไฟล์ชุดข้อมูล Leaf Workshop (หากรันจากไฟล์ .rar)

In [ ]:
!sudo apt-get install -y unrar
import os

rar_path = "../leaf workshop.rar"
if os.path.exists(rar_path) and not os.path.exists("../leaf workshop"):
    print("Extracting dataset archive...")
    !unrar x -inul "../leaf workshop.rar" ../
    print("Done!")
else:
    print("Dataset directory ready!")

## 4. ขั้นตอนที่ 1: สั่งแบ่งข้อมูล Train (70%) / Val (15%) / Test (15%)
เลือกพืชที่ต้องการ: `corn`, `coffee`, `potato`, `banana`, `mango`, `orange`, หรือ `all`

In [ ]:
TARGET_CROP = "corn"  # ตัวเลือก: corn, coffee, potato, banana, mango, orange, all

!python scripts/01_split_dataset.py --crop {TARGET_CROP} --train_ratio 0.70 --val_ratio 0.15 --test_ratio 0.15

## 5. ขั้นตอนที่ 2: เริ่มต้นการฝึกสอนโมเดล (Training on GPU)
เลือกโมเดล Backbone: `mobilenet_v2` (แนะนำสำหรับ Edge AI), `resnet18`, หรือ `leaf_net` (Custom CNN)

In [ ]:
BACKBONE = "mobilenet_v2"  # mobilenet_v2, resnet18, leaf_net
EPOCHS = 15
BATCH_SIZE = 32
LEARNING_RATE = 0.001

!python scripts/02_train.py \
    --crop {TARGET_CROP} \
    --backbone {BACKBONE} \
    --epochs {EPOCHS} \
    --batch_size {BATCH_SIZE} \
    --lr {LEARNING_RATE}

## 6. ดูกราฟประวัติการฝึกสอน (Loss & Accuracy Curves)

In [ ]:
from IPython.display import Image, display
curve_img = f"outputs/reports/{TARGET_CROP}_{BACKBONE}_curves.png"
if os.path.exists(curve_img):
    display(Image(curve_img))
else:
    print("Curve image not found.")

## 7. ขั้นตอนที่ 3: ประเมินผลโมเดลบน Test Set & Confusion Matrix

In [ ]:
checkpoint_file = f"outputs/checkpoints/{TARGET_CROP}_{BACKBONE}_best.pth"

!python scripts/03_evaluate.py --checkpoint {checkpoint_file}

cm_img = f"outputs/reports/{TARGET_CROP}_{BACKBONE}_confusion_matrix.png"
if os.path.exists(cm_img):
    display(Image(cm_img))

## 8. ขั้นตอนที่ 4: ทดสอบทำนายภาพจริง (Inference Playground)

In [ ]:
import glob
# สุ่มภาพตัวอย่างจากชุดทดสอบมาทดลองรัน Inference
test_images = glob.glob(f"../leaf workshop/{TARGET_CROP}*/*/*.jpg")
if test_images:
    sample_img = test_images[0]
    print(f"Testing Image: {sample_img}")
    display(Image(sample_img, width=250))
    !python scripts/04_predict.py --checkpoint {checkpoint_file} --image "{sample_img}"
else:
    print("No sample image found.")

## 9. ขั้นตอนที่ 5: ส่งออกโมเดลเป็น ONNX และ TorchScript สำหรับบอร์ด Edge AI

In [ ]:
!python models/export.py --checkpoint {checkpoint_file}

# ดาวน์โหลดไฟล์โมเดลลงเครื่องคอมพิวเตอร์ของคุณ
from google.colab import files
import glob

for f in glob.glob(f"outputs/exported/{TARGET_CROP}_{BACKBONE}_best*"):
    print(f"Ready to download: {f}")
    # files.download(f)  # ปลด comment เพื่อดาวน์โหลดอัตโนมัติ